# 03 — Evaluate six planner conditions

Three base-model strategies (always ONCE, always SPLIT, prompted adaptive) and three fine-tuned-model conditions (forced ONCE, forced SPLIT, adaptive). All use the same `run_evaluation()` pipeline. Forced conditions are constrained ablations, not the model's natural adaptive policy.


## 1. Configure paths and execution

In [ ]:
from pathlib import Path

GITHUB_REPO = "RanaShoaaib/Adaptive-RAG"
GITHUB_REF = "main"  # Prefer the same commit as notebook 02
INSTALL_FROM_GITHUB = True
LOCAL_REPO = "/content/Adaptive-RAG"
MOUNT_DRIVE = True
DRIVE_ROOT = Path("/content/drive/MyDrive/Adaptive-RAG")
TEST_PATH = None  # Defaults to the tracked test subset. Set a Path to override.
ADAPTER_DIR = DRIVE_ROOT / "results/adaptive_planner_v1/adapter"
RESULTS_DIR = DRIVE_ROOT / "results/heldout_evaluation"

RUN_BASELINES = False  # Base model: always_once, always_split, prompted
RUN_ADAPTIVE = False   # Fine-tuned model: adaptive_once, adaptive_split, adaptive
# Enable either group after a small pilot; each run incurs retrieval/answering cost.
ADAPTIVE_USE_4BIT = True
EXPERIMENT_PREFIX = "test_v1"
SPLIT_BUDGET = 0.05  # Share of questions routed to SPLIT by `adaptive`; matches the teacher label rate (~5%). tau = the (1 - budget) quantile of P(split) on the validation subset


## 2. Install the package

In [ ]:
import subprocess, sys

if INSTALL_FROM_GITHUB:
    checkout = Path(LOCAL_REPO)
    if not checkout.exists():
        subprocess.check_call(["git", "clone", f"https://github.com/{GITHUB_REPO}.git", str(checkout)])
    subprocess.check_call(["git", "-C", str(checkout), "fetch", "origin", GITHUB_REF])
    subprocess.check_call(["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"])
else:
    checkout = Path(LOCAL_REPO)
checkout = checkout.resolve()
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", f"{checkout}[training]"])
sys.path.insert(0, str(checkout))


## 3. Mount Drive, load configuration, and verify held-out questions

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

import torch
from adaptive_rag.config import SLM_MODEL, ANSWERING_MODEL, SEED, FINAL_CONTEXT_TOP_K, TEST_SUBSET_PATH, VAL_SUBSET_PATH
from adaptive_rag.data.prepare_data import load_subset

if RUN_BASELINES or RUN_ADAPTIVE:
    assert torch.cuda.is_available(), "Use a Colab GPU runtime"
TEST_PATH = TEST_SUBSET_PATH if TEST_PATH is None else Path(TEST_PATH)
assert TEST_PATH.is_file(), f"Test subset missing: {TEST_PATH}"
if RUN_ADAPTIVE:
    assert ADAPTER_DIR.is_dir(), f"Adapter missing: {ADAPTER_DIR}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
subset = load_subset(TEST_PATH)
assert len(subset) > 0
ids = [str(row["id"]) for row in subset]
assert len(ids) == len(set(ids)), "Duplicate test question IDs"
print("Configured test path:", TEST_SUBSET_PATH)
print("Actual test path:", TEST_PATH)
print("Questions:", len(subset), "| Seed:", SEED, "| Model:", SLM_MODEL)
print("Answering model:", ANSWERING_MODEL, "| Context top-k:", FINAL_CONTEXT_TOP_K)


## 3b. Set the decision threshold (label-free)

`adaptive` sends a question to SPLIT when P(split) >= tau, where tau is the (1 - `SPLIT_BUDGET`) quantile of P(split) over the external validation subset. No labels or retrieval results are used, and the test set is never used to set tau. The realized test SPLIT rate is reported by the strategy-mix cell and can differ from the budget.


In [ ]:
import gc, json
import numpy as np
from adaptive_rag.planner.adaptive import load_adaptive_planner, score_split_probabilities, threshold_for_budget

DECISION_THRESHOLD = None
THRESHOLD_PATH = RESULTS_DIR / f"{EXPERIMENT_PREFIX}_decision_threshold.json"
if RUN_ADAPTIVE:
    if THRESHOLD_PATH.is_file():
        saved = json.loads(THRESHOLD_PATH.read_text(encoding="utf-8"))
        assert (saved.get("method") == "budget_quantile" and saved["budget"] == SPLIT_BUDGET
                and saved["adapter_dir"] == str(ADAPTER_DIR)), (
            f"{THRESHOLD_PATH} was made with other settings or an older method; move it aside or change EXPERIMENT_PREFIX."
        )
        DECISION_THRESHOLD = saved["threshold"]
        print("Reusing saved threshold:", DECISION_THRESHOLD)
    else:
        val_rows = load_subset(VAL_SUBSET_PATH)
        val_model, val_tokenizer = load_adaptive_planner(
            ADAPTER_DIR, base_model_name=SLM_MODEL, use_4bit=ADAPTIVE_USE_4BIT)
        p_split = score_split_probabilities(val_model, val_tokenizer, [row["question"] for row in val_rows])
        del val_model, val_tokenizer
        gc.collect(); torch.cuda.empty_cache()
        DECISION_THRESHOLD = threshold_for_budget(p_split, SPLIT_BUDGET)
        realized = sum(p >= DECISION_THRESHOLD for p in p_split) / len(p_split)
        THRESHOLD_PATH.write_text(json.dumps({
            "method": "budget_quantile", "threshold": DECISION_THRESHOLD, "budget": SPLIT_BUDGET,
            "n_val": len(p_split), "realized_val_split_rate": realized,
            "p_split_min_median_max": [float(np.min(p_split)), float(np.median(p_split)), float(np.max(p_split))],
            "adapter_dir": str(ADAPTER_DIR), "val_path": str(VAL_SUBSET_PATH),
        }, indent=2), encoding="utf-8")
        print("Saved:", THRESHOLD_PATH)
        print(f"P(split) on validation: min/median/max = {np.round([np.min(p_split), np.median(p_split), np.max(p_split)], 3)}")
        print(f"tau = {DECISION_THRESHOLD:.4f} for a {SPLIT_BUDGET:.0%} budget | realized validation SPLIT rate = {realized:.1%}")


In [ ]:
if RUN_BASELINES or RUN_ADAPTIVE:
    import os
    try:
        from google.colab import userdata
    except ImportError:
        pass  # Local runs use .env or environment variables.
    else:
        # Grant notebook access to the required secrets. Ports/TLS default to 443/true.
        for name in ("WEAVIATE_HTTP_HOST", "WEAVIATE_GRPC_HOST", "WEAVIATE_API_KEY"):
            if not os.getenv(name):
                os.environ[name] = userdata.get(name)
        for name in ("WEAVIATE_HTTP_PORT", "WEAVIATE_HTTP_SECURE", "WEAVIATE_GRPC_PORT", "WEAVIATE_GRPC_SECURE"):
            if name not in os.environ:
                try:
                    os.environ[name] = userdata.get(name)
                except userdata.SecretNotFoundError:
                    pass
    from adaptive_rag.database.weaviate_client import check_retrieval_collection
    print(f"Weaviate ready. Indexed passages: {check_retrieval_collection():,}")


## 4. Evaluate six conditions using `run_evaluation()`

Set `RUN_BASELINES` and `RUN_ADAPTIVE` independently. Use the same held-out test questions and retrieval settings for every run. Invalid planner outputs are saved with `planner_status=failed`; metric means exclude these rows, so compare success rates too.


In [ ]:
from adaptive_rag.evaluation.runner import run_evaluation

BASELINES = ("always_once", "always_split", "prompted")
ADAPTIVE_MODES = ("adaptive_once", "adaptive_split", "adaptive")
STRATEGIES = (*BASELINES, *ADAPTIVE_MODES)
if RUN_ADAPTIVE:
    assert DECISION_THRESHOLD is not None, "The decision threshold is missing; run the section 3b cell with RUN_ADAPTIVE=True first."

for strategy in STRATEGIES:
    is_finetuned = strategy in ADAPTIVE_MODES
    enabled = RUN_ADAPTIVE if is_finetuned else RUN_BASELINES
    if not enabled:
        print(f"Skipped {strategy}")
        continue
    run_evaluation(
        experiment_id=f"{EXPERIMENT_PREFIX}_{strategy}",
        dataset_path=TEST_PATH,
        planner_strategy=strategy,
        sampling_seed=SEED,
        adapter_dir=ADAPTER_DIR if is_finetuned else None,
        adaptive_use_4bit=ADAPTIVE_USE_4BIT if is_finetuned else False,
        decision_threshold=DECISION_THRESHOLD if strategy == "adaptive" else None,
        results_dir=RESULTS_DIR,
    )


## 6. Compare results

Only experiments with saved `results.parquet` are included. Verify all have exactly the same test IDs before interpreting comparisons. The table reports mean metrics and costs, not statistical significance.

In [ ]:
import pandas as pd

strategies = STRATEGIES
frames = {}
for strategy in strategies:
    path = RESULTS_DIR / f"{EXPERIMENT_PREFIX}_{strategy}" / "results.parquet"
    if path.is_file():
        frames[strategy] = pd.read_parquet(path)
    else:
        print("Not found:", path)

if frames:
    expected = set(ids)
    for strategy, df in frames.items():
        assert "id" in df.columns, f"Missing id column: {strategy}"
        assert len(df) == len(expected) and set(df["id"].astype(str)) == expected, (
            f"{strategy} results do not match the complete held-out test subset"
        )
    metric_cols = [
        f"recall_at_{FINAL_CONTEXT_TOP_K}", f"ndcg_at_{FINAL_CONTEXT_TOP_K}",
        "answer_em", "answer_f1", "search_calls", "embedding_calls",
        "reranker_calls", "reranked_pairs", "rewrite_calls",
    ]
    comparison = pd.DataFrame({
        name: {metric: float(df[metric].mean()) for metric in metric_cols}
        for name, df in frames.items()
    }).T
    comparison["planner_success_rate"] = {
        name: float((df["planner_status"] == "ok").mean()) if "planner_status" in df else 1.0
        for name, df in frames.items()
    }
    comparison["planner_retries"] = {
        name: int((df["planner_attempts"] > 1).sum()) if "planner_attempts" in df else 0
        for name, df in frames.items()
    }
    print("Metric means exclude planner failures (NaN). Check success rates alongside them.")
    display(comparison.round(3))
    comparison.to_csv(RESULTS_DIR / f"{EXPERIMENT_PREFIX}_comparison.csv")
else:
    print("No completed experiments found yet.")


## 7. Strategy mix and paired differences

A paired comparison is appropriate because the same questions are used in every experiment. These descriptive differences do not constitute statistical significance testing.

In [ ]:
if frames:
    for name, df in frames.items():
        print(name, "strategy counts:", df["strategy"].value_counts().to_dict())
        if "planner_status" in df:
            print("  planner failures:", int((df["planner_status"] == "failed").sum()))
    if "adaptive" in frames:
        adaptive = frames["adaptive"].set_index("id")
        for baseline in (*BASELINES, "adaptive_once", "adaptive_split"):
            if baseline not in frames:
                continue
            reference = frames[baseline].set_index("id").loc[adaptive.index]
            print(f"Fine-tuned adaptive minus {baseline} (paired successful cases):")
            for metric in [f"recall_at_{FINAL_CONTEXT_TOP_K}", f"ndcg_at_{FINAL_CONTEXT_TOP_K}", "answer_f1", "search_calls"]:
                diff = (adaptive[metric] - reference[metric]).mean()
                print(f"  {metric}: {diff:+.4f}")


## Outputs and interpretation

Six experiment folders, each containing `config.json`, `results.parquet`, `results.csv`, and `summary.json`. The central comparison is `prompted` versus `adaptive`; forced conditions diagnose query generation and strategy selection. `adaptive` uses a fixed SPLIT budget matched to the teacher label rate; compare it with `adaptive_once` to see what that routing adds. The notebook has not been GPU-tested.
